In [ ]:
import cv2
import json
import numpy as np
import tkinter as tk
from tkinter import filedialog
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

model = load_model("models/plant_disease_classifier.keras")

with open("models/plant_class_names.json", "r") as f:
    class_names = json.load(f)

IMG_SIZE = 160

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

choice_window = tk.Toplevel(root)
choice_window.title("Choose Input")
choice_window.attributes('-topmost', True)
choice_window.geometry("320x130")

user_choice = {"mode": None}

def choose_image():
    user_choice["mode"] = "image"
    choice_window.destroy()

def choose_webcam():
    user_choice["mode"] = "webcam"
    choice_window.destroy()

tk.Label(choice_window, text="Diagnose a plant leaf from:").pack(pady=10)
tk.Button(choice_window, text="Upload Image", command=choose_image, width=22).pack(pady=5)
tk.Button(choice_window, text="Use Webcam", command=choose_webcam, width=22).pack(pady=5)

choice_window.wait_window()


def format_label(raw_label):
    parts = raw_label.split("___")
    plant = parts[0].replace("_", " ")
    condition = parts[1].replace("_", " ") if len(parts) > 1 else "Unknown"
    is_healthy = "healthy" in condition.lower()
    return plant, condition, is_healthy


def predict_frame(frame):
    resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    array = preprocess_input(rgb.astype("float32"))
    array = np.expand_dims(array, axis=0)
    prediction = model.predict(array, verbose=0)
    idx = np.argmax(prediction)
    confidence = prediction[0][idx] * 100
    return class_names[idx], confidence


def draw_result(frame, plant, condition, is_healthy, confidence):
    color = (0, 200, 0) if is_healthy else (0, 0, 255)
    status = "Healthy" if is_healthy else "Diseased"
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 90), (30, 30, 30), -1)
    cv2.putText(frame, f"Plant: {plant}", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(frame, f"Status: {status}", (15, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.putText(frame, f"Condition: {condition} ({confidence:.1f}%)", (15, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)


if user_choice["mode"] == "image":
    image_path = filedialog.askopenfilename(
        parent=root,
        title="Select a leaf image",
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
    )
    root.destroy()

    if image_path:
        image = cv2.imread(image_path)
        raw_label, confidence = predict_frame(image)
        plant, condition, is_healthy = format_label(raw_label)

        display = cv2.resize(image, (600, 600))
        draw_result(display, plant, condition, is_healthy, confidence)

        print(f"Plant: {plant}")
        print(f"Status: {'Healthy' if is_healthy else 'Diseased'}")
        print(f"Condition: {condition}")
        print(f"Confidence: {confidence:.1f}%")

        cv2.imshow("Plant Disease Detector", display)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("No image selected.")

elif user_choice["mode"] == "webcam":
    root.destroy()
    cap = cv2.VideoCapture(0)
    print("Press q to quit.")
    frame_count = 0
    last_result = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)

        if frame_count % 10 == 0:
            raw_label, confidence = predict_frame(frame)
            last_result = format_label(raw_label) + (confidence,)

        if last_result:
            plant, condition, is_healthy, confidence = last_result
            draw_result(frame, plant, condition, is_healthy, confidence)

        cv2.imshow("Plant Disease Detector", frame)
        frame_count += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
else:
    root.destroy()
    print("No option selected.")